# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melikekaya01/flyrank-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

I chose this lane because the practical problem is not simply to predict CTR. The useful decision is to identify which already-visible pages deserve a human review first when an SEO or content team has limited time.

The starter dataset contains page-level information such as impressions, clicks, CTR, average position, content type, search intent, content age, freshness, and engagement signals. These variables can help create a position-aware and volume-aware review queue instead of using one simple low-CTR threshold for every page.

The goal is to support a human decision, not to automatically change pages or claim that a model knows why a page performs poorly.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### Research question

Among pseudonymized content pages with enough observed search visibility, which pages should an SEO or content editor review first because their CTR is weak relative to their search position and because supporting content or engagement signals suggest a possible opportunity?

### Unit of analysis

One pseudonymized content page.

### Decision

The decision is which pages should enter the next limited-capacity content or SEO review batch, and in what order.

### Output

The output will be a ranked decision-support list containing an opportunity score, the observed evidence behind the score, and reason codes such as:

* `high_visibility_low_ctr`
* `strong_position_weak_ctr`
* `stale_page`
* `weak_engagement_context`

### Action

A human reviewer can inspect the recommended page and decide whether to:

* improve the title or meta description,
* check search-intent alignment,
* update the content,
* improve snippet structure,
* monitor the page without editing,
* or leave the page unchanged.

### Cost of a wrong recommendation

A false positive may waste editor time and may encourage unnecessary changes to a healthy page. A false negative may leave a real opportunity unreviewed.

Because unnecessary changes can be costly, the project should focus on producing a useful top-ranked review list rather than automatically editing pages.

### Why data or ML can help

CTR is affected by search position, impression volume, content type, intent, age, freshness, and engagement. A single threshold may incorrectly prioritize low-volume or naturally low-position pages.

Data can support a transparent baseline. Machine learning should only be used if it improves the quality of the ranked recommendations compared with that simpler baseline.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
from pathlib import Path
import pandas as pd

possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv")
]

data_path = next((path for path in possible_paths if path.exists()), None)

if data_path is None:
    data_path = "https://raw.githubusercontent.com/melikekaya01/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Total rows:", len(df))
print("Total columns:", df.shape[1])
print("Pseudonymized clients:", df["client_id"].nunique())

Total rows: 30000
Total columns: 44
Pseudonymized clients: 32


In [4]:
eligible = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()

low_ctr_pages = eligible[eligible["ctr"] < 0.5]
zero_click_pages = eligible[eligible["clicks_90d"] == 0]

print("Eligible visible pages:", len(eligible))
print("Pages with CTR below 0.5%:", len(low_ctr_pages))
print("Share with CTR below 0.5%:", round(len(low_ctr_pages) / len(eligible) * 100, 1), "%")
print("Visible pages with zero clicks:", len(zero_click_pages))
print("Total impressions on zero-click pages:", int(zero_click_pages["impressions_90d"].sum()))

Eligible visible pages: 12023
Pages with CTR below 0.5%: 9759
Share with CTR below 0.5%: 81.2 %
Visible pages with zero clicks: 1215
Total impressions on zero-click pages: 1867848


The starter dataset contains **30,000 content pages across 32 pseudonymized clients**.

After applying a conservative evidence filter of at least 500 impressions in 90 days and an observed average position between 1 and 20, there are **12,023 visible pages**.

Among these pages, **9,759 pages, or 81.2%, have an observed CTR below 0.5%**.

There are also **1,215 visible pages with zero clicks**, even though these pages produced approximately **1,867,848 impressions in total**.

These numbers do not prove that the pages are poor or that editing them will improve traffic. However, they show that there is a large and operationally relevant candidate set. This makes it worthwhile to study position adjustment, minimum-volume rules, ranking quality, and false positives during the next seven weeks.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This project can describe observed associations and produce a directional decision-support ranking.

It can evaluate whether a position-aware and volume-aware method places more useful review candidates near the top than a simple baseline.

However, this project cannot prove that a low CTR is caused by a title, meta description, content problem, or a Google algorithm factor.

It also cannot prove that editing a recommended page will cause more clicks, traffic, or engagement. A causal claim would require an experiment or another suitable causal method.

The starter data is a pseudonymized snapshot. Therefore, the results will be described as evidence from this dataset, not as a universal rule about search behavior.

I will also apply the following precautions:

* `avg_position = 0` will be treated as missing position information.
* Low-volume pages will not be treated as if they contain reliable evidence.
* CTR will be interpreted together with average position.
* Percentage columns will be read according to their documented units.
* The recommendations will support human review rather than replace human judgment.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.